In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import rtdl
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import category_encoders as ce

# ✅ (1) PyTorch 환경 변수 설정 (OOM 방지)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# ✅ (2) Device 설정
device = torch.device("cuda" if torch.cuda.is_available())
torch.backends.cudnn.benchmark = True

# ✅ (3) GPU 캐시 정리
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# ✅ (4) 데이터 로드 및 전처리
df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv')

def pre_process(df):
    df = df.loc[:, df.isnull().mean() < 0.8]
    df = df.drop(columns=['ID'])
    df_sol = df['임신 성공 여부']
    df = df.drop(columns=['임신 성공 여부'])

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col].fillna('Unknown', inplace=True)
        else:
            unique_count = df[col].nunique()
            if unique_count <= 4:
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)

    return df, df_sol

pre_df, sol = pre_process(df)

# ✅ (5) 카테고리형 데이터 Embedding (Target Encoding)
category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

encoder = ce.TargetEncoder(cols=category_columns)
X_train, X_valid, y_train, y_valid = train_test_split(pre_df, sol, test_size=0.2, random_state=42, stratify=sol)

X_train[category_columns] = encoder.fit_transform(X_train[category_columns], y_train)
X_valid[category_columns] = encoder.transform(X_valid[category_columns])

# ✅ (6) Tensor 변환 (float16 적용)
X_train_num = torch.tensor(X_train[numeric_columns].values, dtype=torch.float16).to(device)
X_valid_num = torch.tensor(X_valid[numeric_columns].values, dtype=torch.float16).to(device)

if category_columns:
    X_train_cat = torch.tensor(X_train[category_columns].values, dtype=torch.int64).to(device)
    X_valid_cat = torch.tensor(X_valid[category_columns].values, dtype=torch.int64).to(device)
else:
    X_train_cat = None
    X_valid_cat = None

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float16).to(device)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float16).to(device)

# ✅ (7) FT-Transformer 모델 생성 (메모리 절약: n_blocks=2)
cat_cardinalities = [len(df[col].unique()) for col in category_columns] if category_columns else []

model = rtdl.FTTransformer.make_baseline(
    n_num_features=len(numeric_columns),
    cat_cardinalities=cat_cardinalities,
    last_layer_query_idx=[-1],
    d_out=1,

    d_token=32,
    n_blocks=1,  # ✅ n_blocks=3 → 2 (OOM 방지)
    attention_dropout=0.1,
    ffn_d_hidden=32,  # ✅ 128 → 64 (메모리 절약)
    ffn_dropout=0.1,
    residual_dropout=0.025
).to(device)

# ✅ (8) 모든 GPU 사용
if torch.cuda.device_count() > 1:
    print(f"🔥 Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# ✅ (9) 옵티마이저 & Learning Rate Scheduler 추가
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)

# ✅ (10) 손실 함수 (Focal Loss 적용)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return loss.mean()

loss_fn = FocalLoss(alpha=0.5, gamma=2.0)

# ✅ (11) 학습 실행 (배치 크기 줄이고, Gradient Accumulation 적용)
batch_size = 4  # ✅ 16 → 8 (OOM 방지)
gradient_accumulation_steps = 2  # ✅ 2 스텝마다 업데이트

best_auc = 0.0
patience = 50
counter = 0
epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()

    # ✅ Gradient Accumulation 적용
    for i in range(gradient_accumulation_steps):
        with torch.cuda.amp.autocast():
            output = model(X_train_num, X_train_cat)
            loss = loss_fn(output.squeeze(), y_train_tensor) / gradient_accumulation_steps  # ✅ loss 축소

        loss.backward()

    optimizer.step()

    # ✅ 검증
    with torch.no_grad():
        y_pred_proba = torch.sigmoid(model(X_valid_num, X_valid_cat)).cpu().numpy().squeeze()
        auc = roc_auc_score(y_valid.cpu().numpy(), y_pred_proba)

    scheduler.step(auc)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f} - AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

# ✅ (12) 모델 평가
model.eval()
if torch.cuda.device_count() > 1:
    model = model.module

with torch.no_grad():
    y_pred_proba = torch.sigmoid(model(X_valid_num, X_valid_cat)).cpu().numpy().squeeze()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    
# y_valid는 pandas.Series이므로 .cpu()를 호출할 수 없음
# 대신 y_valid_tensor를 사용해야 함
y_valid_np = y_valid_tensor.numpy().astype(int)  # float16 → int 변환

roc_auc = roc_auc_score(y_valid_np, y_pred_proba)
acc = accuracy_score(y_valid_np, y_pred)
f1 = f1_score(y_valid_np, y_pred, average='macro')
recall = recall_score(y_valid_np, y_pred, average='macro')
precision = precision_score(y_valid_np, y_pred, average='macro')

print(f"✅ FT-Transformer Accuracy: {acc:.4f}")
print(f"🚀 FT-Transformer F1 Score: {f1:.4f}")
print(f"🚀 FT-Transformer Recall: {recall:.4f}")
print(f"🚀 FT-Transformer Precision: {precision:.4f}")
print(f"🚀 FT-Transformer ROC-AUC: {roc_auc:.4f}")




🔥 Using 4 GPUs!


AttributeError: 'Series' object has no attribute 'cpu'